### 1. Configuration

Monthly **Placement × Post Site Visit** analysis.

**Placement** = `placements.name` only  
(e.g. **Brand Phrase**, **Brand Phrase Overseas**, Brand Broad, Display, FIM Forms, …)  
— not CID, not URL.

Join path:
`leads.utm_campaign` (CID) → `campaigntrackers.placement` → `placements.name`

Excel: **two sheets**
1. `By_URL_CID` — Month × Placement × URL × CID (bifurcation)
2. `By_Placement` — Month × Placement overall (Brand Phrase, Overseas, …)


In [ ]:
from __future__ import annotations

from datetime import date, timedelta
from pathlib import Path
from typing import Any

import awswrangler as wr
import boto3
import numpy as np
import pandas as pd
from IPython.display import display
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows

ANALYSIS_START_DATE = date(2026, 1, 1)
ANALYSIS_END_DATE = date.today()

PSV_STAGE_OVERRIDE: str | None = None
PLACEMENT_COL_OVERRIDE: str | None = None

OUTPUT_XLSX = Path("/Users/lekhanigam/data/leads_sv_closure_reports/Placement_Monthly_Analysis.xlsx")
ENV_CANDIDATES = [Path("/Users/lekhanigam/data/.env"), Path(".env")]

START_TS = f"TIMESTAMP '{ANALYSIS_START_DATE.isoformat()} 00:00:00'"
END_EXCL = ANALYSIS_END_DATE + timedelta(days=1)
END_TS_EXCL = f"TIMESTAMP '{END_EXCL.isoformat()} 00:00:00'"

print("Window:", ANALYSIS_START_DATE, "→", ANALYSIS_END_DATE)
print("Output:", OUTPUT_XLSX)


### 2. Athena Connection


In [ ]:
def _aws_session(env: dict) -> "boto3.Session":
    """SSO profile — run: aws sso login --profile <AWS_PROFILE>"""
    profile = (env.get("AWS_PROFILE") or "asbl").strip()
    region = (env.get("AWS_REGION") or "ap-south-1").strip()
    session = boto3.Session(profile_name=profile, region_name=region)
    try:
        session.client("sts").get_caller_identity()
    except Exception as e:
        raise RuntimeError(
            f"AWS SSO login required. Run:\n  aws sso login --profile {profile}\n({e})"
        ) from e
    return session


def _load_env(path: Path) -> dict[str, str]:
    out: dict[str, str] = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        out[k.strip()] = v.strip()
    return out


_env: dict[str, str] = {}
for _p in ENV_CANDIDATES:
    if _p.exists():
        _env = _load_env(_p)
        print("Loaded", _p)
        break
else:
    raise FileNotFoundError("No .env found")

AWS_REGION = _env.get("AWS_REGION", "ap-south-1")
ATHENA_DATABASE = _env.get("ATHENA_DATABASE", "leadsdb-prod")
ATHENA_WORKGROUP = _env.get("ATHENA_WORKGROUP", "primary")
ATHENA_S3_OUTPUT = _env.get("ATHENA_OUTPUT_LOCATION", "s3://metabase-asbl/athena-query-results/")

AWS_PROFILE = _env.get("AWS_PROFILE", "asbl")
session = _aws_session(_env)
print("AWS profile:", AWS_PROFILE)
print("AWS account:", session.client("sts").get_caller_identity().get("Account"))


def athena_query(sql: str) -> pd.DataFrame:
    return wr.athena.read_sql_query(
        sql=sql,
        database=ATHENA_DATABASE,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_S3_OUTPUT,
        ctas_approach=False,
    )


probe = athena_query("SELECT 1 AS ok FROM leads LIMIT 1")
assert not probe.empty, "Cannot read leads — check credentials / workgroup / database"
print("Athena OK |", ATHENA_DATABASE, "|", ATHENA_WORKGROUP)


### 3. Identify / Verify Relevant Table

Probe `leads`, `leadstagehistories`, `campaigntrackers`, `placements`, `campaignurls`, `visitorlogs`.
Confirm Post Site Visit stage and placement names (Brand Phrase, Display, …).


In [ ]:
def _norm(s: str) -> str:
    return "".join(ch for ch in str(s).lower() if ch.isalnum())


def probe(table: str) -> pd.DataFrame | None:
    try:
        df = athena_query(f"SELECT * FROM {table} LIMIT 1")
        return df if df is not None and len(df.columns) else None
    except Exception as e:
        print(f"  skip {table}: {type(e).__name__}: {str(e)[:120]}")
        return None


def pick_col(cols: list[str], synonyms: list[str]) -> str | None:
    ncol = {_norm(c): c for c in cols}
    for s in synonyms:
        if _norm(s) in ncol:
            return ncol[_norm(s)]
    for s in synonyms:
        hits = sorted([c for c in cols if _norm(s) in _norm(c)], key=len)
        if hits:
            return hits[0]
    return None


LEAD_TABLE = "leads"
STAGE_TABLE = "leadstagehistories"
assert probe(LEAD_TABLE) is not None, "leads not readable"
assert probe(STAGE_TABLE) is not None, "leadstagehistories not readable"

lead_sample = probe(LEAD_TABLE)
stage_sample = probe(STAGE_TABLE)
lead_cols = list(lead_sample.columns)
stage_cols = list(stage_sample.columns)

TRACKER_TABLE = "campaigntrackers" if probe("campaigntrackers") is not None else None
PLACEMENTS_TABLE = "placements" if probe("placements") is not None else None
CAMPAIGNURLS_TABLE = "campaignurls" if probe("campaignurls") is not None else None
VISITORLOGS_TABLE = "visitorlogs" if probe("visitorlogs") is not None else None

print("LEAD:", LEAD_TABLE)
print("STAGE:", STAGE_TABLE)
print("TRACKER:", TRACKER_TABLE)
print("PLACEMENTS:", PLACEMENTS_TABLE)
print("CAMPAIGNURLS:", CAMPAIGNURLS_TABLE)
print("VISITORLOGS:", VISITORLOGS_TABLE)

assert TRACKER_TABLE and PLACEMENTS_TABLE, "Need campaigntrackers + placements"

tracker_cols = list(probe(TRACKER_TABLE).columns)
place_cols = list(probe(PLACEMENTS_TABLE).columns)
curl_cols = list(probe(CAMPAIGNURLS_TABLE).columns) if CAMPAIGNURLS_TABLE else []

COL = {
    "lead_pk": pick_col(lead_cols, ["_id", "id"]),
    "lead_id": pick_col(lead_cols, ["leadid", "lead_id"]),
    "cid": pick_col(lead_cols, ["utm_campaign", "cid", "campaignid"]),
    "visitor_id": pick_col(lead_cols, ["visitorid", "visitor_id"]),
    "stage_lead": pick_col(stage_cols, ["lead", "leadid"]),
    "stage_name": pick_col(stage_cols, ["stage"]),
    "stage_created": pick_col(stage_cols, ["createdat", "created_at"]),
    "stage_pk": pick_col(stage_cols, ["_id", "id"]),
    "ct_cid": pick_col(tracker_cols, ["cid"]),
    "ct_placement": pick_col(tracker_cols, ["placement"]),
    "ct_campaignurl": pick_col(tracker_cols, ["campaignurl", "campaign_url"]),
    "ct_customurl": pick_col(tracker_cols, ["customurl", "custom_url"]),
    "ct_updated": pick_col(tracker_cols, ["updatedat", "updated_at", "createdat"]),
    "p_pk": pick_col(place_cols, ["_id", "id"]),
    "p_name": pick_col(place_cols, ["name", "placementname"]),
    "cu_cid": pick_col(curl_cols, ["cid"]) if curl_cols else None,
    "cu_baseurl": pick_col(curl_cols, ["baseurl", "base_url", "url"]) if curl_cols else None,
}

print("Resolved fields:")
for k, v in COL.items():
    print(f"  {k:16s} → {v}")

need = [
    "lead_pk", "lead_id", "cid", "stage_lead", "stage_name", "stage_created",
    "ct_cid", "ct_placement", "p_pk", "p_name",
]
missing = [k for k in need if not COL[k]]
if missing:
    raise RuntimeError(f"Missing critical fields: {missing}")

stage_values = athena_query(f"""
SELECT CAST({COL['stage_name']} AS varchar) AS stage, COUNT(*) AS n
FROM {STAGE_TABLE}
GROUP BY 1 ORDER BY n DESC
""")
print("\nDistinct stages:")
display(stage_values.head(20))

if PSV_STAGE_OVERRIDE:
    PSV_STAGE = PSV_STAGE_OVERRIDE
else:
    ranked = []
    for _, r in stage_values.iterrows():
        s = str(r["stage"]).strip().lower()
        score = (5 if ("post" in s and "site" in s) else 0) + (1 if "visit" in s else 0) - (4 if "pre" in s else 0)
        ranked.append((score, int(r["n"]), str(r["stage"])))
    ranked.sort(reverse=True)
    if not ranked or ranked[0][0] <= 0:
        raise RuntimeError("Could not detect Post Site Visit — set PSV_STAGE_OVERRIDE")
    PSV_STAGE = ranked[0][2]
print("PSV stage:", repr(PSV_STAGE))

place_preview = athena_query(f"""
SELECT
  COALESCE(NULLIF(TRIM(CAST(p.{COL['p_name']} AS varchar)), ''), 'NA') AS placement,
  COUNT(*) AS n_trackers,
  COUNT(DISTINCT CAST(ct.{COL['ct_cid']} AS varchar)) AS n_cid
FROM {TRACKER_TABLE} ct
LEFT JOIN {PLACEMENTS_TABLE} p
  ON CAST(ct.{COL['ct_placement']} AS varchar) = CAST(p.{COL['p_pk']} AS varchar)
GROUP BY 1
ORDER BY n_trackers DESC
LIMIT 30
""")
print("\nTop placements (campaigntrackers → placements):")
display(place_preview)

url_preview = athena_query(f"""
SELECT
  COALESCE(
    NULLIF(TRIM(CAST(ct.{COL['ct_customurl']} AS varchar)), ''),
    NULLIF(TRIM(CAST(ct.{COL['ct_campaignurl']} AS varchar)), ''),
    'NA'
  ) AS url,
  COUNT(*) AS n
FROM {TRACKER_TABLE} ct
GROUP BY 1
ORDER BY n DESC
LIMIT 15
""")
print("\nSample tracker URLs:")
display(url_preview)

PLACEMENT_SOURCE = (
    f"leads.{COL['cid']} → {TRACKER_TABLE}.cid → {PLACEMENTS_TABLE}.name; "
    "URL = coalesce(customurl, campaignurl, campaignurls.baseurl)"
)
print("\nPlacement/URL source:", PLACEMENT_SOURCE)


### 4. Build Monthly Placement-Level SV Query

**Placement** = `placements.name` (Brand Phrase, Brand Phrase Overseas, …).  
If CID has no tracker/placement match → `Unmapped`.

URL = `coalesce(customurl, campaignurl, campaignurls.baseurl)`  
CID = `leads.utm_campaign`

**SV count** = `COUNT(DISTINCT lead_id)` at earliest PSV month.  
Athena grain: **month × placement × url × cid**.


In [ ]:
psv_sql = str(PSV_STAGE).replace("'", "''")
stage_pk_expr = COL["stage_pk"] or COL["stage_lead"]
ct_order = COL["ct_updated"] or COL["ct_cid"]

url_from_cu = (
    f"NULLIF(TRIM(CAST(cu.{COL['cu_baseurl']} AS varchar)), '')"
    if COL["cu_cid"] and COL["cu_baseurl"] and CAMPAIGNURLS_TABLE
    else "CAST(NULL AS varchar)"
)
cu_join = ""
if COL["cu_cid"] and COL["cu_baseurl"] and CAMPAIGNURLS_TABLE:
    cu_join = f"""
  LEFT JOIN (
    SELECT
      CAST({COL['cu_cid']} AS varchar) AS cid,
      {COL['cu_baseurl']} AS baseurl,
      ROW_NUMBER() OVER (PARTITION BY CAST({COL['cu_cid']} AS varchar) ORDER BY 1) AS rn
    FROM {CAMPAIGNURLS_TABLE}
  ) cu
    ON cu.cid = CAST(l.{COL['cid']} AS varchar) AND cu.rn = 1
"""

# Placement = placements.name ONLY (Brand Phrase, Brand Phrase Overseas, …)
placement_expr = f"""
COALESCE(
  NULLIF(TRIM(CAST(p.{COL['p_name']} AS varchar)), ''),
  'Unmapped'
)
"""

url_expr = f"""
COALESCE(
  NULLIF(TRIM(CAST(ct.customurl AS varchar)), ''),
  NULLIF(TRIM(CAST(ct.campaignurl AS varchar)), ''),
  {url_from_cu},
  'NA'
)
"""

cid_expr = f"""
COALESCE(NULLIF(TRIM(CAST(l.{COL['cid']} AS varchar)), ''), 'NA')
"""

SQL = f"""
WITH psv AS (
  SELECT
    CAST(h.{COL['stage_lead']} AS varchar) AS lead_oid,
    h.{COL['stage_created']} AS event_ts,
    ROW_NUMBER() OVER (
      PARTITION BY CAST(h.{COL['stage_lead']} AS varchar)
      ORDER BY h.{COL['stage_created']} ASC, CAST(h.{stage_pk_expr} AS varchar) ASC
    ) AS rn
  FROM {STAGE_TABLE} h
  WHERE CAST(h.{COL['stage_name']} AS varchar) = '{psv_sql}'
    AND h.{COL['stage_created']} >= {START_TS}
    AND h.{COL['stage_created']} < {END_TS_EXCL}
),
first_psv AS (
  SELECT lead_oid, event_ts FROM psv WHERE rn = 1
),
ct_one AS (
  SELECT *
  FROM (
    SELECT
      CAST(ct.{COL['ct_cid']} AS varchar) AS cid,
      ct.{COL['ct_placement']} AS placement_fk,
      ct.{COL['ct_customurl']} AS customurl,
      ct.{COL['ct_campaignurl']} AS campaignurl,
      ROW_NUMBER() OVER (
        PARTITION BY CAST(ct.{COL['ct_cid']} AS varchar)
        ORDER BY ct.{ct_order} DESC
      ) AS rn
    FROM {TRACKER_TABLE} ct
  ) x
  WHERE rn = 1
),
enriched AS (
  SELECT
    date_format(f.event_ts, '%Y-%m') AS month_key,
    {placement_expr} AS placement,
    {url_expr} AS url,
    {cid_expr} AS cid,
    CAST(l.{COL['lead_id']} AS varchar) AS lead_id
  FROM first_psv f
  INNER JOIN {LEAD_TABLE} l
    ON CAST(l.{COL['lead_pk']} AS varchar) = f.lead_oid
  LEFT JOIN ct_one ct
    ON ct.cid = CAST(l.{COL['cid']} AS varchar)
  LEFT JOIN {PLACEMENTS_TABLE} p
    ON CAST(ct.placement_fk AS varchar) = CAST(p.{COL['p_pk']} AS varchar)
  {cu_join}
)
SELECT
  month_key,
  placement,
  url,
  cid,
  COUNT(DISTINCT lead_id) AS sv_count
FROM enriched
GROUP BY 1, 2, 3, 4
ORDER BY 1, 2, 3, 4
"""

print("Placement/URL source:", PLACEMENT_SOURCE)
print(SQL)


### 5. Run Query


In [ ]:
agg = athena_query(SQL)
print("Aggregated rows (month × placement × url × cid):", len(agg))
display(agg.head(25))

val = athena_query(f"""
WITH psv AS (
  SELECT
    CAST(h.{COL['stage_lead']} AS varchar) AS lead_oid,
    ROW_NUMBER() OVER (
      PARTITION BY CAST(h.{COL['stage_lead']} AS varchar)
      ORDER BY h.{COL['stage_created']} ASC
    ) AS rn
  FROM {STAGE_TABLE} h
  WHERE CAST(h.{COL['stage_name']} AS varchar) = '{psv_sql}'
    AND h.{COL['stage_created']} >= {START_TS}
    AND h.{COL['stage_created']} < {END_TS_EXCL}
)
SELECT COUNT(*) AS unique_psv_leads FROM psv WHERE rn = 1
""")
UNIQUE_PSV_LEADS = int(val.iloc[0]["unique_psv_leads"])
print("Unique PSV leads in window:", UNIQUE_PSV_LEADS)


### 6. Validate Aggregated Data


In [ ]:
agg = agg.copy()
agg["sv_count"] = pd.to_numeric(agg["sv_count"], errors="coerce").fillna(0).astype(int)

for c in ["placement", "url", "cid", "month_key"]:
    agg[c] = agg[c].astype("string").fillna("")
    blank = agg[c].str.strip().isin(["", "nan", "None", "<NA>", "NA"])
    if c == "placement":
        agg.loc[blank, c] = "Unmapped"
    else:
        agg.loc[blank, c] = "NA"

place_month = (
    agg.groupby(["month_key", "placement"], dropna=False)["sv_count"].sum().reset_index()
)
placement_total = int(place_month["sv_count"].sum())
detail_total = int(agg["sv_count"].sum())
n_placements = int(agg["placement"].nunique())
n_urls = int(agg["url"].nunique())
n_cids = int(agg["cid"].nunique())
n_months = int(agg["month_key"].nunique())

checks = []


def check(name, ok, detail=""):
    checks.append({"check": name, "status": "PASS" if ok else "FAIL", "detail": detail})


check(
    "unique_psv_vs_detail_sum",
    UNIQUE_PSV_LEADS == detail_total,
    f"unique={UNIQUE_PSV_LEADS}, detail_sum={detail_total}",
)
check(
    "place_month_sum_eq_detail",
    placement_total == detail_total,
    f"place_month={placement_total}, detail={detail_total}",
)
check("no_null_month", bool(agg["month_key"].notna().all()))
check("has_rows", len(agg) > 0)
check(
    "has_named_placements",
    bool((agg["placement"] != "Unmapped").any()),
    "expected Brand Phrase / similar from placements.name",
)

VALIDATION_STATUS = "FAIL" if any(c["status"] == "FAIL" for c in checks) else "PASS"
print("Validation:", VALIDATION_STATUS)
display(pd.DataFrame(checks))
print(
    f"Unique PSV leads: {UNIQUE_PSV_LEADS} | Placements: {n_placements} | "
    f"URLs: {n_urls} | CIDs: {n_cids} | Months: {n_months}"
)
print("Min/Max month:", agg["month_key"].min(), "→", agg["month_key"].max())
print("Top placements:")
display(
    place_month.groupby("placement", as_index=False)["sv_count"]
    .sum()
    .sort_values("sv_count", ascending=False)
    .head(15)
)


### 7. Build Sheet Frames

1. **By_URL_CID** — Month × Placement × URL × CID  
2. **By_Placement** — Month × **Placement name** (Brand Phrase, Brand Phrase Overseas, …)


In [ ]:
def month_label(key: str) -> str:
    try:
        return pd.Period(key, freq="M").strftime("%b-%y")
    except Exception:
        return str(key)


# Sheet 1: URL / CID bifurcation (still carries Placement name)
detail_out = agg.copy()
detail_out["Month"] = detail_out["month_key"].map(month_label)
detail_out = detail_out.rename(
    columns={
        "placement": "Placement",
        "url": "URL",
        "cid": "CID",
        "sv_count": "SV Count",
    }
)[["Month", "Placement", "URL", "CID", "SV Count"]]
detail_out = detail_out.sort_values(
    ["Month", "Placement", "URL", "CID"]
).reset_index(drop=True)

# Sheet 2: overall by Placement name (Brand Phrase, Overseas, …)
pivot = place_month.pivot_table(
    index="month_key",
    columns="placement",
    values="sv_count",
    aggfunc="sum",
    fill_value=0,
).sort_index()

# Put Unmapped last; order other placements by total SV desc
totals = pivot.sum(axis=0).sort_values(ascending=False)
cols = [c for c in totals.index if c != "Unmapped"]
if "Unmapped" in pivot.columns:
    cols = cols + ["Unmapped"]
pivot = pivot.reindex(columns=cols).astype(int)
pivot["Grand Total"] = pivot.sum(axis=1)

pivot_out = pivot.copy()
pivot_out.index = [month_label(m) for m in pivot_out.index]
pivot_out.index.name = "Month"
grand = pivot_out.sum(axis=0).astype(int)
grand.name = "Grand Total"
pivot_out = pd.concat([pivot_out, pd.DataFrame([grand])])

# Long form too (handy check): Month | Placement | SV
place_long = place_month.copy()
place_long["Month"] = place_long["month_key"].map(month_label)
place_long = place_long.rename(columns={"placement": "Placement", "sv_count": "SV Count"})
place_long = place_long[["Month", "Placement", "SV Count"]].sort_values(
    ["Month", "SV Count", "Placement"], ascending=[True, False, True]
).reset_index(drop=True)

gt = int(pivot_out.loc["Grand Total", "Grand Total"])
print("Placement pivot Grand Total:", gt, "| Unique PSV:", UNIQUE_PSV_LEADS, "| match:", gt == UNIQUE_PSV_LEADS)
if gt != UNIQUE_PSV_LEADS:
    VALIDATION_STATUS = "FAIL"

print("\nPlacement names in data (top):")
display(place_long.groupby("Placement", as_index=False)["SV Count"].sum().sort_values("SV Count", ascending=False).head(25))
print("\nBy_URL_CID (head):")
display(detail_out.head(20))
print("\nBy_Placement pivot:")
display(pivot_out)


### 8. Export Excel (2 sheets)


In [ ]:
HEADER_FILL = PatternFill("solid", fgColor="1F4E79")
HEADER_FONT = Font(color="FFFFFF", bold=True)
TITLE_FONT = Font(bold=True, size=14, color="1F4E79")
GT_FILL = PatternFill("solid", fgColor="D9E2F3")
GT_FONT = Font(bold=True)
THIN = Border(
    left=Side(style="thin", color="B0B0B0"),
    right=Side(style="thin", color="B0B0B0"),
    top=Side(style="thin", color="B0B0B0"),
    bottom=Side(style="thin", color="B0B0B0"),
)


def style_sheet(ws, df, title: str, subtitle: str):
    ws.cell(1, 1, title).font = TITLE_FONT
    ws.cell(2, 1, subtitle)
    header_row = 4
    for r_i, row in enumerate(dataframe_to_rows(df, index=False, header=True), start=header_row):
        for c_i, val in enumerate(row, start=1):
            cell = ws.cell(r_i, c_i, val)
            cell.border = THIN
            cell.alignment = Alignment(
                vertical="center", horizontal="center" if c_i > 1 else "left"
            )
            is_header = r_i == header_row
            is_gt_row = (not is_header) and len(row) and str(row[0]) == "Grand Total"
            col_name = df.columns[c_i - 1]
            is_gt_col = col_name == "Grand Total"
            if is_header:
                cell.fill = HEADER_FILL
                cell.font = HEADER_FONT
            elif is_gt_row or is_gt_col:
                cell.fill = GT_FILL
                cell.font = GT_FONT
            if (not is_header) and isinstance(val, (int, float)) and col_name != "Month":
                if col_name in {"SV Count", "Grand Total"} or col_name in set(pivot_out.columns):
                    cell.number_format = "#,##0"
    for col_idx in range(1, df.shape[1] + 1):
        vals = [
            len(str(ws.cell(r, col_idx).value or ""))
            for r in range(header_row, header_row + len(df) + 1)
        ]
        col_name = str(df.columns[col_idx - 1])
        max_w = 48 if col_name in {"URL", "CID"} else 28
        width = min(max(max(vals) + 2, 10), max_w)
        letter = get_column_letter(col_idx)
        ws.column_dimensions[letter].width = width
    ws.auto_filter.ref = f"A{header_row}:{get_column_letter(df.shape[1])}{header_row + len(df)}"
    ws.freeze_panes = f"A{header_row + 1}"


subtitle = (
    f"Period: {ANALYSIS_START_DATE} → {ANALYSIS_END_DATE} | Stage: {PSV_STAGE!r} | "
    f"Placement = placements.name (Brand Phrase, Overseas, …) | SV = COUNT(DISTINCT lead_id)"
)

wb = Workbook()

ws_detail = wb.active
ws_detail.title = "By_URL_CID"
style_sheet(
    ws_detail,
    detail_out,
    "Bifurcation: Placement × URL × CID",
    subtitle,
)

ws_place = wb.create_sheet("By_Placement")
style_sheet(
    ws_place,
    pivot_out.reset_index(),
    "Overall by Placement (Brand Phrase, Brand Phrase Overseas, …)",
    subtitle,
)

wb.save(OUTPUT_XLSX)
print("Wrote", OUTPUT_XLSX)
print("Sheets:", wb.sheetnames)
print("Placement columns (sample):", list(pivot_out.columns)[:12], "…")
assert wb.sheetnames == ["By_URL_CID", "By_Placement"]


### 9. Final Validation


In [ ]:
print("Analysis Period:", ANALYSIS_START_DATE, "to", ANALYSIS_END_DATE)
print("Total Post Site Visit Leads:", UNIQUE_PSV_LEADS)
print("Unique Placements:", n_placements)
print("Unique URLs:", n_urls)
print("Unique CIDs:", n_cids)
print("Monthly Records (month×placement×url×cid):", len(agg))
print("Output:", OUTPUT_XLSX)
print("Sheets: By_URL_CID, By_Placement")
print("Validation:", VALIDATION_STATUS)
print("Placement/URL source:", PLACEMENT_SOURCE)
assert OUTPUT_XLSX.exists()
print("DONE")
